# Lab 02: Data Loading & Transformation

## Objectives
- Load data from various file formats (CSV, JSON, Parquet)
- Understand schema inference vs explicit schema definition
- Perform data cleaning and validation
- Handle missing values and nulls
- Transform data using built-in functions
- Write data to different formats

## Domain Coverage
- **Data Loading — 10%**
- **DataFrame API — 30%**

## Estimates
- **Time:** ~60-90 minutes
- **Difficulty:** Beginner
- **Prerequisites:** Lab 01 (Spark Fundamentals)

![Data Loading Diagram](../assets/diagrams/lab-02-data-loading.mmd)

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, coalesce, regexp_replace, upper, lower, trim
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

In [ ]:
# Create Spark session
spark = SparkSession.builder \
    .appName("DataLoadingTransformation") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

# Verify Spark session
print(f"Spark version: {spark.version}")
print(f"Spark master: {spark.conf.get('spark.master')}")

## A. Loading CSV Data

CSV is one of the most common data formats. Spark provides flexible options for reading CSV files.

In [ ]:
# Load CSV with schema inference
df_csv = spark.read.csv("data/sample/employees.csv", header=True, inferSchema=True)

print("CSV loaded with schema inference:")
df_csv.printSchema()
df_csv.show()

In [ ]:
# Load CSV with explicit schema
explicit_schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("department", StringType(), True),
    StructField("salary", DoubleType(), True)
])

df_csv_explicit = spark.read.csv("data/sample/employees.csv", header=True, schema=explicit_schema)

print("CSV loaded with explicit schema:")
df_csv_explicit.printSchema()

## B. Loading JSON Data

JSON is a semi-structured format that can handle nested data and complex types.

In [ ]:
# Load JSON data
df_json = spark.read.json("data/sample/employees.json")

print("JSON data loaded:")
df_json.printSchema()
df_json.show(truncate=False)

In [ ]:
# Handle nested arrays in JSON
from pyspark.sql.functions import explode

df_exploded = df_json.withColumn("skill", explode(col("skills")))
df_exploded.select("name", "skill").show()

## C. Loading Parquet Data

Parquet is a columnar storage format optimized for Spark workloads.

In [ ]:
# First, write some data to Parquet
df_csv.write.mode("overwrite").parquet("data/parquet/employees.parquet")

# Load Parquet data
df_parquet = spark.read.parquet("data/parquet/employees.parquet")

print("Parquet data loaded:")
df_parquet.printSchema()
df_parquet.show()

## D. Data Cleaning and Validation

Real-world data often requires cleaning. Let's practice common data cleaning operations.

In [ ]:
# Create a DataFrame with null values
data_with_nulls = [
    ("Alice", 25, "Engineering", 75000),
    ("Bob", None, "Marketing", None),
    ("Charlie", 35, None, 85000),
    (None, 28, "Sales", 60000),
    ("Eve", 32, "Marketing", 70000)
]

df_nulls = spark.createDataFrame(data_with_nulls, ["name", "age", "department", "salary"])
print("DataFrame with null values:")
df_nulls.show()

In [ ]:
# Handle null values
# Drop rows with any null values
df_no_nulls = df_nulls.na.drop()
print("Drop rows with any nulls:")
df_no_nulls.show()

# Fill null values with defaults
df_filled = df_nulls.na.fill({"age": 0, "salary": 50000, "department": "Unknown", "name": "Anonymous"})
print("\nFill null values with defaults:")
df_filled.show()

In [ ]:
# Use coalesce for conditional null handling
df_coalesce = df_nulls.withColumn(
    "clean_salary",
    coalesce(col("salary"), lit(50000))
)
print("Using coalesce for conditional null handling:")
df_coalesce.show()

## E. Data Transformation

Transform data using built-in functions for common operations.

In [ ]:
# String transformations
df_string = df_csv.withColumn("name_upper", upper(col("name"))) \
                   .withColumn("name_lower", lower(col("name"))) \
                   .withColumn("name_trimmed", trim(col("name")))

print("String transformations:")
df_string.select("name", "name_upper", "name_lower", "name_trimmed").show()

In [ ]:
# Numeric transformations
df_numeric = df_csv.withColumn("salary_annual", col("salary") * 12) \
                   .withColumn("salary_with_bonus", col("salary") * 1.10) \
                   .withColumn("age_squared", col("age") ** 2)

print("Numeric transformations:")
df_numeric.select("name", "salary", "salary_annual", "salary_with_bonus", "age_squared").show()

In [ ]:
# Regular expression replacements
# Create data with messy strings
messy_data = [("Alice-123", "Engineering"), ("Bob_456", "Marketing"), ("Charlie 789", "Sales")]
df_messy = spark.createDataFrame(messy_data, ["name_id", "department"])

# Remove digits from name
df_clean = df_messy.withColumn("name_clean", regexp_replace(col("name_id"), "\\d", "")) \
                      .withColumn("name_clean", regexp_replace(col("name_clean"), "[-_]", " "))

print("Regular expression cleaning:")
df_clean.show()

## F. Data Validation

Validate data to ensure quality and consistency.

In [ ]:
# Check for data quality issues
print("Data validation checks:")

# Check for null values
print(f"\nNull values per column:")
for column in df_csv.columns:
    null_count = df_csv.filter(col(column).isNull()).count()
    print(f"{column}: {null_count}")

# Check for duplicates
total_rows = df_csv.count()
distinct_rows = df_csv.distinct().count()
print(f"\nTotal rows: {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Duplicates: {total_rows - distinct_rows}")

In [ ]:
# Filter invalid data
# Remove rows with invalid age (negative or > 100)
df_valid = df_csv.filter((col("age") > 0) & (col("age") < 100))

# Remove rows with invalid salary (negative)
df_valid = df_valid.filter(col("salary") > 0)

print(f"Original rows: {df_csv.count()}")
print(f"Valid rows: {df_valid.count()}")

## G. Writing Data

Write transformed data to various formats for persistence.

In [ ]:
# Write to different formats
df_valid.write.mode("overwrite").csv("data/csv/employees_clean.csv")
df_valid.write.mode("overwrite").json("data/json/employees_clean.json")
df_valid.write.mode("overwrite").parquet("data/parquet/employees_clean.parquet")

print("Data written to CSV, JSON, and Parquet formats")

In [ ]:
# Demonstrate different write modes
# Append mode
df_valid.write.mode("append").parquet("data/parquet/employees_append.parquet")

# Ignore mode (don't write if data exists)
df_valid.write.mode("ignore").parquet("data/parquet/employees_append.parquet")

# Error if exists mode
try:
    df_valid.write.mode("errorifexists").parquet("data/parquet/employees_append.parquet")
except Exception as e:
    print(f"Expected error with errorifexists mode: {type(e).__name__}")

print("\nWrite modes demonstrated")

## H. Partitioning Data

Partition data when writing for better query performance.

In [ ]:
# Write partitioned by department
df_valid.write.mode("overwrite") \
       .partitionBy("department") \
       .parquet("data/parquet/employees_partitioned")

print("Data partitioned by department")

# Read partitioned data
df_partitioned = spark.read.parquet("data/parquet/employees_partitioned")
print(f"\nPartitioned data rows: {df_partitioned.count()}")
df_partitioned.show()

## Exam-Style Review

**Q1.** What is the difference between schema inference and explicit schema definition?
- A) Schema inference is faster, explicit schema is slower
- B) Schema inference lets Spark guess types, explicit schema defines types manually
- C) They are identical
- D) Explicit schema is only for JSON files

**Q2.** Which function is used to handle null values conditionally?
- A) fill()
- B) drop()
- C) coalesce()
- D) replace()

**Q3.** What is the advantage of Parquet format over CSV?
- A) Parquet is human-readable
- B) Parquet is columnar and optimized for Spark
- C) Parquet supports more data types
- D) Parquet is always smaller in size

**Q4.** Which write mode will fail if the data already exists?
- A) overwrite
- B) append
- C) ignore
- D) errorifexists

### Answers
- **Q1: B** — Schema inference lets Spark guess data types, while explicit schema defines types manually for better control.
- **Q2: C** — coalesce() returns the first non-null value, useful for conditional null handling.
- **Q3: B** — Parquet is a columnar storage format optimized for Spark workloads with better compression and query performance.
- **Q4: D** — errorifexists mode will fail if the data already exists at the destination.

## Key Takeaways
- Spark supports multiple data formats: CSV, JSON, Parquet, and more
- Schema inference is convenient but explicit schema provides better control
- Handle null values with drop(), fill(), and coalesce()
- Transform data using built-in functions for strings, numbers, and regex
- Validate data to ensure quality and consistency
- Write data in different modes: overwrite, append, ignore, errorifexists
- Partition data by columns for better query performance

**Next:** [Lab 03 — Advanced DataFrame Operations](chapter-03-advanced-dataframe-operations.ipynb)

In [ ]:
# Cleanup
spark.catalog.clearCache()
print("Lab 02 complete. Cache cleared.")